# fMRI Denoising Pipeline - Example Usage

This notebook demonstrates how to use the denoising pipeline to extract time-series from BOLD fMRI data.

In [1]:
import sys
sys.path.append('..')  # Add parent directory to path

from denoising import DenoisingPipeline
from denoising.config import load_config
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

## 1. Load Configuration

First, load the configuration file that defines atlas, denoising, and confounds parameters.

In [2]:
config = load_config('../configs/strategy_4.yaml')

print(f"Atlas: {config.atlas.name} with {config.atlas.n_regions} regions")
print(f"Smoothing FWHM: {config.denoising.smoothing_fwhm} mm")
print(f"Standardization: {config.denoising.standardize}")

Atlas: brainnetome with 400 regions
Smoothing FWHM: 4.0 mm
Standardization: False


In [ ]:
config

## 2. Initialize Pipeline

Create a `DenoisingPipeline` instance with the configuration.

In [3]:
pipeline = DenoisingPipeline(config)
print("Pipeline initialized successfully")

Pipeline initialized successfully


## 3. Process a Single Subject

Specify paths to BOLD and confounds files, then run the pipeline.

In [4]:
from nilearn.interfaces.fmriprep import load_confounds
strategy_4 = {'strategy': ['motion', 'compcor', 'high_pass', 'global_signal'],
                      'motion': 'full',
                      'compcor': 'anat_combined',
                      'n_compcor': 10,
                      'global_signal': 'full',
                      }
bold_path = r"/data/Projects/ABIDE2/ABIDEII-BNI_1/derivatives/sub-29006/ses-1/func/sub-29006_ses-1_task-rest_run-1_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz"
df, _ = load_confounds(bold_path, **strategy_4)

In [5]:
df

,a_comp_cor_00,a_comp_cor_01,a_comp_cor_02,a_comp_cor_03,a_comp_cor_04,a_comp_cor_05,a_comp_cor_06,a_comp_cor_07,a_comp_cor_08,a_comp_cor_09,...,trans_x_derivative1_power2,trans_x_power2,trans_y,trans_y_derivative1,trans_y_derivative1_power2,trans_y_power2,trans_z,trans_z_derivative1,trans_z_derivative1_power2,trans_z_power2
0,-3.361346e-12,-4.201680e-12,8.403339e-13,-3.032122e-18,3.361343e-12,2.521007e-12,-6.722690e-12,-3.061277e-18,2.521011e-12,2.521011e-12,...,-0.001624,-0.001314,-0.029934,0.091180,0.002067,0.000404,0.054405,-0.030716,-0.045440,-0.003651
1,3.197925e-02,-6.933164e-02,1.051619e-02,-8.785900e-02,1.243728e-01,1.141994e-01,-6.087445e-02,1.002494e-01,-3.651477e-03,7.837478e-02,...,-0.001624,-0.001380,0.062052,0.091180,0.002067,-0.004123,0.023494,-0.030716,-0.045440,-0.012271
2,-8.073318e-03,-7.708244e-02,-1.095965e-02,3.361358e-02,7.570140e-02,9.458547e-02,-5.253556e-02,1.848136e-01,8.479587e-02,1.161813e-02,...,-0.001628,-0.001414,0.012339,-0.050519,-0.003923,-0.003778,0.036536,0.013238,-0.046225,-0.008867
3,6.863872e-02,-5.361280e-02,-1.779601e-02,-2.174908e-03,1.441385e-01,7.331948e-02,-3.754979e-02,6.853433e-02,8.478374e-02,6.629715e-02,...,-0.001596,-0.001494,-0.028314,-0.041458,-0.004742,0.000178,0.048472,0.012130,-0.046253,-0.005454
4,2.046444e-02,-2.323847e-02,-4.371579e-02,1.411343e-03,9.520917e-02,1.483349e-01,-6.085878e-02,6.257384e-02,5.030497e-02,3.447155e-03,...,-0.001617,-0.001509,0.062351,0.089859,0.001825,-0.004110,0.059946,0.011669,-0.046264,-0.001904
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,-2.411272e-02,-1.590354e-02,1.496370e-01,3.206513e-03,-1.219618e-01,9.488769e-03,-1.974977e-01,6.041507e-02,2.767851e-02,2.143242e-01,...,-0.001090,-0.001499,-0.082532,-0.193965,0.030916,0.010598,0.121903,0.370395,0.090652,0.021813
116,9.750682e-02,1.368976e-01,1.440044e-01,-1.483609e-01,6.544443e-02,-6.680180e-02,-1.047169e-01,-2.114481e-02,1.127436e-01,-1.758598e-02,...,-0.001157,-0.000888,-0.090916,-0.009189,-0.006325,0.012734,0.196383,0.074675,-0.040848,0.060485
117,-5.815560e-02,-1.066511e-01,-3.450737e-02,-7.328114e-02,-7.085210e-02,3.689764e-02,-1.289006e-01,-8.295238e-02,1.103018e-02,1.565325e-02,...,-0.001611,-0.001086,0.034141,0.124252,0.009244,-0.004537,-0.200458,-0.396646,0.111087,-0.017641
118,-4.057433e-02,8.773976e-02,8.667739e-02,-8.672244e-02,5.964598e-02,8.043439e-02,4.803687e-02,-7.879178e-02,8.369385e-02,-6.859915e-02,...,-0.001032,0.000519,-0.035349,-0.070296,-0.001566,0.001198,0.038429,0.239082,0.010672,-0.008345


In [6]:
from denoising.io.confounds import ConfoundsHandler
conf = ConfoundsHandler(pipeline.config.confounds)

In [7]:
confounds_path = r"/data/Projects/ABIDE2/ABIDEII-BNI_1/derivatives/sub-29006/ses-1/func/sub-29006_ses-1_task-rest_run-1_desc-confounds_timeseries.tsv"

cc = conf.load_and_select(confounds_path)

In [12]:
cc.interpolate(inplace=True)

,a_comp_cor_00,a_comp_cor_01,a_comp_cor_02,a_comp_cor_03,a_comp_cor_04,a_comp_cor_05,a_comp_cor_06,a_comp_cor_07,a_comp_cor_08,a_comp_cor_09,...,trans_x_derivative1_power2,trans_x_power2,trans_y,trans_y_derivative1,trans_y_derivative1_power2,trans_y_power2,trans_z,trans_z_derivative1,trans_z_derivative1_power2,trans_z_power2
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,NaN,1.943846e-04,-0.070599,NaN,NaN,0.004984,0.154877,NaN,NaN,0.023987
1,0.031979,-0.069332,0.010516,-0.087859,0.124373,0.114199,-0.060874,0.100249,-0.003651,0.078375,...,0.000007,1.285258e-04,0.021387,0.091986,0.008461,0.000457,0.123966,-0.030911,0.000955,0.015368
2,-0.008073,-0.077082,-0.010960,0.033614,0.075701,0.094585,-0.052536,0.184814,0.084796,0.011618,...,0.000003,9.490658e-05,-0.028327,-0.049713,0.002471,0.000802,0.137008,0.013043,0.000170,0.018771
3,0.068639,-0.053613,-0.017796,-0.002175,0.144138,0.073319,-0.037550,0.068534,0.084784,0.066297,...,0.000034,1.499383e-05,-0.068979,-0.040652,0.001653,0.004758,0.148944,0.011935,0.000142,0.022184
4,0.020464,-0.023238,-0.043716,0.001411,0.095209,0.148335,-0.060859,0.062574,0.050305,0.003447,...,0.000014,3.713777e-08,0.021686,0.090665,0.008220,0.000470,0.160418,0.011474,0.000132,0.025734
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,-0.024113,-0.015904,0.149637,0.003207,-0.121962,0.009489,-0.197498,0.060415,0.027679,0.214324,...,0.000541,1.001506e-05,-0.123198,-0.193159,0.037310,0.015178,0.222375,0.370199,0.137048,0.049451
116,0.097507,0.136898,0.144004,-0.148361,0.065444,-0.066802,-0.104717,-0.021145,0.112744,-0.017586,...,0.000473,6.205477e-04,-0.131581,-0.008384,0.000070,0.017314,0.296855,0.074480,0.005547,0.088123
117,-0.058156,-0.106651,-0.034507,-0.073281,-0.070852,0.036898,-0.128901,-0.082952,0.011030,0.015653,...,0.000019,4.226892e-04,-0.006524,0.125057,0.015639,0.000043,-0.099986,-0.396841,0.157483,0.009997
118,-0.040574,0.087740,0.086677,-0.086722,0.059646,0.080434,0.048037,-0.078792,0.083694,-0.068599,...,0.000599,2.027426e-03,-0.076015,-0.069491,0.004829,0.005778,0.138901,0.238887,0.057067,0.019294


In [11]:
cc.isna().sum()

a_comp_cor_00                       0
a_comp_cor_01                       0
a_comp_cor_02                       0
a_comp_cor_03                       0
a_comp_cor_04                       0
a_comp_cor_05                       0
a_comp_cor_06                       0
a_comp_cor_07                       0
a_comp_cor_08                       0
a_comp_cor_09                       0
cosine00                            0
cosine01                            0
cosine02                            0
cosine03                            0
global_signal                       0
global_signal_derivative1           1
global_signal_derivative1_power2    1
global_signal_power2                0
rot_x                               0
rot_x_derivative1                   1
rot_x_derivative1_power2            1
rot_x_power2                        0
rot_y                               0
rot_y_derivative1                   1
rot_y_derivative1_power2            1
rot_y_power2                        0
rot_z       

In [ ]:
import numpy as np
strategy_4 = {'strategy': ['motion', 'compcor', 'high_pass', 'global_signal'],
                'motion': 'full',
                'compcor': 'anat_combined',
                'n_compcor': 10,
                'global_signal': 'full',}

bold_path = r"/data/Projects/ABIDE2/ABIDEII-BNI_1/derivatives/sub-29006/ses-1/func/sub-29006_ses-1_task-rest_run-1_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz"
confounds_path = r"/data/Projects/ABIDE2/ABIDEII-BNI_1/derivatives/sub-29006/ses-1/func/sub-29006_ses-1_task-rest_run-1_desc-confounds_timeseries.tsv"

df_nilearn, _ = load_confounds(bold_path, **strategy_4)

config = load_config('/home/tm/projects/Denoising/configs/strategy_4.yaml')
conf = ConfoundsHandler(config.confounds)

df_conf = conf.load_and_select(confounds_path)

assert df_nilearn.isna().values.sum() == 0
a=df_conf.isna().values.sum() == 0

#print(df_nilearn.values[1:, :].round(4))
#print(df_conf.values[1:, :].round(4))



[120 120 120 120 120 120 120 120 120 120 120 120 120 120   0   0   0   0
   0 119 119 120   0 119 119 120   0 119 119 120   0 119   0   0   0   0
   0   0   0   0   0   0]


In [30]:
a = np.isclose(df_nilearn.values, df_conf.values, rtol=1e-03, atol=1e-03)

print(sum(a))

[120 120 120 120 120 120 120 120 120 120 120 120 120 120   0   0   0   0
   0 119 119 120 120 119 119 120 120 119 119 120   0 119   0   0   0 119
   0   0   0 119   0   0]


In [34]:
df_nilearn.columns.to_numpy()[sum(a)==0]

array(['global_signal', 'global_signal_derivative1',
       'global_signal_derivative1_power2', 'global_signal_power2',
       'rot_x', 'trans_x', 'trans_x_derivative1_power2', 'trans_x_power2',
       'trans_y', 'trans_y_derivative1_power2', 'trans_y_power2',
       'trans_z', 'trans_z_derivative1_power2', 'trans_z_power2'],
      dtype=object)

In [35]:
df_nilearn[df_nilearn.columns.to_numpy()[sum(a)==0]]

,global_signal,global_signal_derivative1,global_signal_derivative1_power2,global_signal_power2,rot_x,trans_x,trans_x_derivative1_power2,trans_x_power2,trans_y,trans_y_derivative1_power2,trans_y_power2,trans_z,trans_z_derivative1_power2,trans_z_power2
0,24.436850,-15.926623,248.347853,48796.903326,0.000378,-0.006894,-0.001624,-0.001314,-0.029934,0.002067,0.000404,0.054405,-0.045440,-0.003651
1,8.294658,-15.926623,248.347853,16417.271410,0.001001,-0.009499,-0.001624,-0.001380,0.062052,0.002067,-0.004123,0.023494,-0.045440,-0.012271
2,5.173509,-2.905580,-2.480944,10216.680481,0.001997,-0.011094,-0.001628,-0.001414,0.012339,-0.003923,-0.003778,0.036536,-0.046225,-0.008867
3,4.160231,-0.797709,-11.195784,8207.854885,0.001973,-0.016964,-0.001596,-0.001494,-0.028314,-0.004742,0.000178,0.048472,-0.046253,-0.005454
4,2.889678,-1.054985,-10.608209,5691.880866,0.001428,-0.020644,-0.001617,-0.001509,0.062351,0.001825,-0.004110,0.059946,-0.046264,-0.001904
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,0.876278,6.334731,25.221631,1711.521223,-0.000603,-0.017672,-0.001090,-0.001499,-0.082532,0.030916,0.010598,0.121903,0.090652,0.021813
116,-2.691720,-3.352430,0.508098,-5322.265416,-0.001228,0.004074,-0.001157,-0.000888,-0.090916,-0.006325,0.012734,0.196383,-0.040848,0.060485
117,-0.360953,2.546336,-6.790038,-730.382126,0.000753,-0.000277,-0.001611,-0.001086,0.034141,0.009244,-0.004537,-0.200458,0.111087,-0.017641
118,-2.212809,-1.636288,-8.793142,-4379.642073,-0.000173,0.024191,-0.001032,0.000519,-0.035349,-0.001566,0.001198,0.038429,0.010672,-0.008345


In [36]:
df_conf[df_nilearn.columns.to_numpy()[sum(a)==0]]

,global_signal,global_signal_derivative1,global_signal_derivative1_power2,global_signal_power2,rot_x,trans_x,trans_x_derivative1_power2,trans_x_power2,trans_y,trans_y_derivative1_power2,trans_y_power2,trans_z,trans_z_derivative1_power2,trans_z_power2
0,1011.021363,NaN,NaN,1.022164e+06,-0.000675,0.013942,NaN,1.943846e-04,-0.070599,NaN,0.004984,0.154877,NaN,0.023987
1,994.879171,-16.142192,260.570368,9.897846e+05,-0.000052,0.011337,0.000007,1.285258e-04,0.021387,0.008461,0.000457,0.123966,0.000955,0.015368
2,991.758022,-3.121149,9.741572,9.835840e+05,0.000944,0.009742,0.000003,9.490658e-05,-0.028327,0.002471,0.000802,0.137008,0.000170,0.018771
3,990.744744,-1.013278,1.026731,9.815751e+05,0.000920,0.003872,0.000034,1.499383e-05,-0.068979,0.001653,0.004758,0.148944,0.000142,0.022184
4,989.474191,-1.270553,1.614306,9.790592e+05,0.000375,0.000193,0.000014,3.713777e-08,0.021686,0.008220,0.000470,0.160418,0.000132,0.025734
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,987.460792,6.119162,37.444147,9.750788e+05,-0.001656,0.003165,0.000541,1.001506e-05,-0.123198,0.037310,0.015178,0.222375,0.137048,0.049451
116,983.892793,-3.567999,12.730613,9.680450e+05,-0.002281,0.024911,0.000473,6.205477e-04,-0.131581,0.000070,0.017314,0.296855,0.005547,0.088123
117,986.223561,2.330768,5.432477,9.726369e+05,-0.000300,0.020559,0.000019,4.226892e-04,-0.006524,0.015639,0.000043,-0.099986,0.157483,0.009997
118,984.371704,-1.851857,3.429373,9.689877e+05,-0.001226,0.045027,0.000599,2.027426e-03,-0.076015,0.004829,0.005778,0.138901,0.057067,0.019294


In [25]:
df_conf

,a_comp_cor_00,a_comp_cor_01,a_comp_cor_02,a_comp_cor_03,a_comp_cor_04,a_comp_cor_05,a_comp_cor_06,a_comp_cor_07,a_comp_cor_08,a_comp_cor_09,...,trans_x_derivative1_power2,trans_x_power2,trans_y,trans_y_derivative1,trans_y_derivative1_power2,trans_y_power2,trans_z,trans_z_derivative1,trans_z_derivative1_power2,trans_z_power2
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,NaN,1.943846e-04,-0.070599,NaN,NaN,0.004984,0.154877,NaN,NaN,0.023987
1,0.031979,-0.069332,0.010516,-0.087859,0.124373,0.114199,-0.060874,0.100249,-0.003651,0.078375,...,0.000007,1.285258e-04,0.021387,0.091986,0.008461,0.000457,0.123966,-0.030911,0.000955,0.015368
2,-0.008073,-0.077082,-0.010960,0.033614,0.075701,0.094585,-0.052536,0.184814,0.084796,0.011618,...,0.000003,9.490658e-05,-0.028327,-0.049713,0.002471,0.000802,0.137008,0.013043,0.000170,0.018771
3,0.068639,-0.053613,-0.017796,-0.002175,0.144138,0.073319,-0.037550,0.068534,0.084784,0.066297,...,0.000034,1.499383e-05,-0.068979,-0.040652,0.001653,0.004758,0.148944,0.011935,0.000142,0.022184
4,0.020464,-0.023238,-0.043716,0.001411,0.095209,0.148335,-0.060859,0.062574,0.050305,0.003447,...,0.000014,3.713777e-08,0.021686,0.090665,0.008220,0.000470,0.160418,0.011474,0.000132,0.025734
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,-0.024113,-0.015904,0.149637,0.003207,-0.121962,0.009489,-0.197498,0.060415,0.027679,0.214324,...,0.000541,1.001506e-05,-0.123198,-0.193159,0.037310,0.015178,0.222375,0.370199,0.137048,0.049451
116,0.097507,0.136898,0.144004,-0.148361,0.065444,-0.066802,-0.104717,-0.021145,0.112744,-0.017586,...,0.000473,6.205477e-04,-0.131581,-0.008384,0.000070,0.017314,0.296855,0.074480,0.005547,0.088123
117,-0.058156,-0.106651,-0.034507,-0.073281,-0.070852,0.036898,-0.128901,-0.082952,0.011030,0.015653,...,0.000019,4.226892e-04,-0.006524,0.125057,0.015639,0.000043,-0.099986,-0.396841,0.157483,0.009997
118,-0.040574,0.087740,0.086677,-0.086722,0.059646,0.080434,0.048037,-0.078792,0.083694,-0.068599,...,0.000599,2.027426e-03,-0.076015,-0.069491,0.004829,0.005778,0.138901,0.238887,0.057067,0.019294


In [25]:
cc[1, 1], df.iloc[1, 1]

(np.float64(-0.0693316354), np.float64(-0.06933163540420167))

In [5]:
# Update these paths to your data
bold_path = r"/data/Projects/ABIDE2/ABIDEII-BNI_1/derivatives/sub-29006/ses-1/func/sub-29006_ses-1_task-rest_run-1_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz"
confounds_path = r"/data/Projects/ABIDE2/ABIDEII-BNI_1/derivatives/sub-29006/ses-1/func/sub-29006_ses-1_task-rest_run-1_desc-confounds_timeseries.tsv"
output_dir = "../output"

# Run the pipeline
output_file = pipeline.process_subject(bold_path, confounds_path, output_dir)
print(f"Output saved to: {output_file}")

ValueError: array must not contain infs or NaNs

## 4. Load and Inspect Results

Load the extracted time-series and explore the data.

In [ ]:
# Load the time-series
timeseries = pd.read_csv(output_file)
print(f"Time-series shape: {timeseries.shape}")
print(f"\nFirst 5 rows:")
timeseries.head()

In [ ]:
# Display region names
print("Brain regions:")
for i, region in enumerate(timeseries.columns[:10]):  # Show first 10
    print(f"  {i+1}. {region}")
print(f"  ... and {len(timeseries.columns) - 10} more")

## 5. Visualize Time-Series

Plot time-series for a few regions.

In [ ]:
# Plot first 5 regions
fig, axes = plt.subplots(5, 1, figsize=(12, 10), sharex=True)

for i, ax in enumerate(axes):
    region = timeseries.columns[i]
    ax.plot(timeseries[region].values)
    ax.set_ylabel('Signal')
    ax.set_title(region)

axes[-1].set_xlabel('Timepoint')
plt.tight_layout()
plt.show()

## 6. Correlation Matrix

Compute and visualize the correlation matrix between regions.

In [ ]:
# Compute correlation matrix (subset of regions for visualization)
n_regions_plot = 50
corr_matrix = timeseries.iloc[:, :n_regions_plot].corr()

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='RdBu_r', center=0, vmin=-1, vmax=1, 
            square=True, cbar_kws={'label': 'Correlation'})
plt.title(f'Correlation Matrix (First {n_regions_plot} Regions)')
plt.tight_layout()
plt.show()

## 7. Batch Processing

Process multiple subjects using a list of file paths.

In [ ]:
# Define subjects to process
subjects = [
    {
        "bold_path": "path/to/subject1_bold.nii.gz",
        "confounds_path": "path/to/subject1_confounds.tsv"
    },
    {
        "bold_path": "path/to/subject2_bold.nii.gz",
        "confounds_path": "path/to/subject2_confounds.tsv"
    }
]

# Run batch processing
# outputs = pipeline.process_batch(subjects, output_dir)
# print(f"Processed {len([o for o in outputs if o])} subjects successfully")

## 8. Custom Configuration

You can modify the configuration for different analysis needs.

In [ ]:
# Example: Create a custom config with different atlas
from denoising.config.schemas import PipelineConfig

custom_config = PipelineConfig(
    atlas={
        "name": "schaefer_2018",
        "resolution": 1,
        "n_regions": 200
    },
    denoising={
        "smoothing_fwhm": 8.0,
        "detrend": True,
        "standardize": "zscore"
    },
    confounds={
        "strategy": "simple",
        "columns": []
    }
)

print(f"Custom config: {custom_config.atlas.n_regions} regions at {custom_config.atlas.resolution}mm")